In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('CIC-DS-FULL.csv')
df

,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Mean,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,ClassLabel
0,4,2,0,12.0,0.0,6.0,6.000000,0.000000,0.0,0.00000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
1,1,2,0,12.0,0.0,6.0,6.000000,0.000000,0.0,0.00000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
2,3,2,0,12.0,0.0,6.0,6.000000,0.000000,0.0,0.00000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
3,1,2,0,12.0,0.0,6.0,6.000000,0.000000,0.0,0.00000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
4,609,7,4,484.0,414.0,233.0,69.142860,111.967896,207.0,103.50000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9167576,44797921,6,5,36.0,0.0,6.0,6.000000,0.000000,0.0,0.00000,...,23663.5,108.36512,23727.0,23502.0,10000000.0,72.038765,10000000.0,10000000.0,Benign,Benign
9167577,49,3,0,76.0,0.0,45.0,25.333334,23.028967,0.0,0.00000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
9167578,1286687,41,42,2664.0,6954.0,456.0,64.975610,109.864570,976.0,165.57143,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
9167579,217,2,1,31.0,6.0,31.0,15.500000,21.920311,6.0,6.00000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign


In [7]:
df['Label'].value_counts()

Label
Benign                  7186189
DDoS-LOIC-HTTP           575364
DoS-Hulk                 318740
DDoS-HOIC                198861
Botnet                   145968
DDoS                     128062
DDoS-NTP                 121328
DDoS-TFTP                 98833
Bruteforce-SSH            97260
Infiltration              94857
DoS-Goldeneye             52324
DDoS-Syn                  47757
DDoS-UDP                  28863
DoS-Slowloris             15243
DDoS-MSSQL                11784
DDoS-UDPLag                8452
Bruteforce-FTP             5984
DoS-Slowhttptest           5271
DDoS-Ddossim               5115
DDoS-DNS                   3668
DoS-Slowread               2786
Portscan                   2255
DDoS-LDAP                  2092
Webattack-bruteforce       2020
DDoS-SNMP                  2017
DDoS-Slowloris             1858
DoS-Slowheaders            1649
Webattack-XSS               876
DoS-Rudy                    699
DDoS-NetBIOS                675
DoS-Slowbody                621
We

In [8]:
df['ClassLabel'].value_counts()

ClassLabel
Benign          7186189
DDoS            1234729
DoS              397344
Botnet           145968
Bruteforce       103244
Infiltration      94857
Webattack          2995
Portscan           2255
Name: count, dtype: int64

In [18]:
# ==========================================
# 🚀 IMPORTS
# ==========================================
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# ==========================================
# 📂 LOAD DATASET
# ==========================================
df = pd.read_csv("CIC-DS-FULL.csv")

print("Original Shape:", df.shape)

# ==========================================
# 🎯 STEP 1: CREATE BINARY LABEL
# ==========================================
df['BinaryLabel'] = (df['ClassLabel'] != 'Benign').astype(int)

# ==========================================
# 📉 STEP 2: REDUCE DATASET (1.1M)
# ==========================================
df, _ = train_test_split(
    df,
    train_size=1100000,
    stratify=df['BinaryLabel'],
    random_state=42
)

print("Reduced Shape:", df.shape)

# ==========================================
# 🔥 STEP 3: GLOBAL SPLIT FIRST (NO LEAKAGE)
# ==========================================
df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df['BinaryLabel'],
    random_state=42
)

print("Train Shape:", df_train.shape)
print("Test Shape :", df_test.shape)

# ==========================================
# 🧹 STEP 4: SPLIT FEATURES & LABEL
# ==========================================
drop_cols = ['Label', 'ClassLabel', 'BinaryLabel']

X_train = df_train.drop(columns=drop_cols)
y_train = df_train['BinaryLabel']

X_test = df_test.drop(columns=drop_cols)
y_test = df_test['BinaryLabel']

# ==========================================
# 🧪 STEP 5: CLEAN DATA (FIT ON TRAIN ONLY)
# ==========================================
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_train = X_train.fillna(X_train.median())

X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.fillna(X_train.median())  # ⚠️ use TRAIN stats

# ==========================================
# ⚖️ STEP 6: STANDARDIZATION (TRAIN ONLY)
# ==========================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# ==========================================
# 📊 STEP 7: FEATURE SELECTION (TRAIN ONLY)
# ==========================================
k = 25

selector = SelectKBest(score_func=mutual_info_classif, k=k)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

# Get selected feature names
selected_features = X_train.columns[selector.get_support()]

print("\nSelected Features:")
print(list(selected_features))

# Convert to DataFrames
df_train_selected = pd.DataFrame(X_train_selected, columns=selected_features)
df_test_selected = pd.DataFrame(X_test_selected, columns=selected_features)

# ==========================================
# 📦 STEP 8: FINAL DATAFRAMES
# ==========================================
df_train_final = df_train_selected.copy()
df_train_final['BinaryLabel'] = y_train.values

df_test_final = df_test_selected.copy()
df_test_final['BinaryLabel'] = y_test.values

print("\nFinal Train Shape:", df_train_final.shape)
print("Final Test Shape :", df_test_final.shape)

# ==========================================
# 💾 STEP 9: SAVE
# ==========================================
df_train_final.to_csv("processed_train.csv", index=False)
df_test_final.to_csv("processed_test.csv", index=False)

print("\n✅ Leak-free preprocessing complete!")

Original Shape: (9167581, 59)
Reduced Shape: (1100000, 60)
Train Shape: (880000, 60)
Test Shape : (220000, 60)

Selected Features:
['Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Max', 'Fwd Header Length', 'Bwd Header Length', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Bytes', 'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes']

Final Train Shape: (880000, 26)
Final Test Shape : (220000, 26)

✅ Leak-free preprocessing complete!


In [1]:
# 11 April 2026

In [3]:
# ==========================================
# 🚀 IMPORTS
# ==========================================
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ==========================================
# 📂 LOAD DATASET
# ==========================================
df = pd.read_csv("CIC-DS-FULL.csv")

print("Original Shape:", df.shape)

# ==========================================
# 🎯 STEP 1: CREATE BINARY LABEL
# ==========================================
df['BinaryLabel'] = (df['ClassLabel'] != 'Benign').astype(int)

# ==========================================
# 📉 STEP 2: REDUCE DATASET (1.1M)
# ==========================================
df, _ = train_test_split(
    df,
    train_size=1100000,
    stratify=df['BinaryLabel'],
    random_state=42
)

print("Reduced Shape:", df.shape)

# ==========================================
# 🔥 STEP 3: GLOBAL SPLIT FIRST (NO LEAKAGE)
# ==========================================
df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df['BinaryLabel'],
    random_state=42
)

print("Train Shape:", df_train.shape)
print("Test Shape :", df_test.shape)

# ==========================================
# 🧹 STEP 4: SPLIT FEATURES & LABEL
# ==========================================
drop_cols = ['Label', 'ClassLabel', 'BinaryLabel']

X_train = df_train.drop(columns=drop_cols)
y_train = df_train['BinaryLabel']

X_test = df_test.drop(columns=drop_cols)
y_test = df_test['BinaryLabel']

# ==========================================
# 🧪 STEP 5: CLEAN DATA (TRAIN-ONLY FIT)
# ==========================================
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_train = X_train.fillna(X_train.median())

X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.fillna(X_train.median())  # use TRAIN stats

# ==========================================
# ⚖️ STEP 6: STANDARDIZATION (TRAIN ONLY)
# ==========================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame (keep feature names)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("\nNumber of Features Used:", X_train_scaled.shape[1])

# ==========================================
# 📦 STEP 7: FINAL DATAFRAMES (ALL FEATURES)
# ==========================================
df_train_final = X_train_scaled.copy()
df_train_final['BinaryLabel'] = y_train.values

df_test_final = X_test_scaled.copy()
df_test_final['BinaryLabel'] = y_test.values

print("\nFinal Train Shape:", df_train_final.shape)
print("Final Test Shape :", df_test_final.shape)

# ==========================================
# 💾 STEP 8: SAVE
# ==========================================
df_train_final.to_csv("processed_train.csv", index=False)
df_test_final.to_csv("processed_test.csv", index=False)

print("\n✅ Leak-free preprocessing complete (ALL FEATURES USED)!")

Original Shape: (9167581, 59)
Reduced Shape: (1100000, 60)
Train Shape: (880000, 60)
Test Shape : (220000, 60)

Number of Features Used: 57

Final Train Shape: (880000, 58)
Final Test Shape : (220000, 58)

✅ Leak-free preprocessing complete (ALL FEATURES USED)!


In [4]:
df_test_final = pd.read_csv('old-processing-splitting/processed_test.csv')
df_train_final = pd.read_csv('old-processing-splitting/processed_train.csv')

In [5]:
print(df_train_final.columns.equals(df_test_final.columns))

True
